# 03b - Baseline: a pretrained generalist model used as it is

Question: how good is a large pretrained model (Cellpose-SAM) on our validation images **used as it is, without us training it**, compared with our own U-Net trained on the training split?

A warning about fairness: Cellpose-SAM was trained on the Cellpose nuclei set among others, and that set contains BBBC038 images with corrected masks (Stringer et al. 2021; Pachitariu et al. 2025). So the model has probably seen many of these validation images before. Its score here is an upper bound, not a clean zero-shot result.

Same validation images, same split file, same metrics as notebook 03 (Data Science Bowl AP over IoU 0.5 to 0.95, F1 at IoU 0.5, merged and split nuclei per image type). Nothing in this notebook trains a model.

Kaggle setup: Accelerator GPU T4, Internet on. Optional: **+ Add Input > Your Work > the saved version of notebook 03**, then the last section compares both models image by image.
Output: `out/per_image_cellpose.csv`, `out/by_type_cellpose.csv`, `out/compare.csv` (if notebook 03 was added), zipped as `results_03b.zip`.

In [ ]:
!pip -q install cellpose==4.0.3

In [ ]:
import glob, zipfile, urllib.request, io, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io as skio
import torch
from cellpose import models

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('.')
DATA = Path('/tmp/nuclei_data')
OUT = WORK / 'out'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)
print('GPU:', torch.cuda.is_available())

URL = 'https://data.broadinstitute.org/bbbc/BBBC038/stage1_train.zip'
zip_path, train_dir = DATA / 'stage1_train.zip', DATA / 'stage1_train'
if not zip_path.exists():
    urllib.request.urlretrieve(URL, zip_path)
if not train_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(train_dir)

SPLIT_URL = 'https://raw.githubusercontent.com/veliatees/nuclei-segmentation/main/results/split.csv'
split = pd.read_csv(io.BytesIO(urllib.request.urlopen(SPLIT_URL).read()))
paths = {Path(p).stem: p for p in glob.glob(str(train_dir / '**' / 'images' / '*.png'), recursive=True)}
val = split[(split.split == 'val') & split.id.isin(paths)].reset_index(drop=True)
print(val.type.value_counts())

In [ ]:
def load(img_path):
    img = skio.imread(img_path)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]
    label = np.zeros(img.shape[:2], np.int32)
    for k, mp in enumerate(sorted(glob.glob(str(Path(img_path).parent.parent / 'masks' / '*.png'))), start=1):
        label[skio.imread(mp) > 0] = k
    return img, label

## The scoring functions

Exactly the same functions as in notebook 03, so the numbers are comparable.

In [ ]:
import numpy as np
from scipy import ndimage as ndi
from skimage.segmentation import watershed, relabel_sequential


def remove_small(label, min_size):
    """Drop labelled objects smaller than min_size pixels and renumber 1..n."""
    sizes = np.bincount(label.ravel())
    small = sizes < min_size
    small[0] = False
    label = label.copy()
    label[small[label]] = 0
    return relabel_sequential(label)[0]


def to_instances(prob, thr=0.5, min_size=15):
    """Pixel probabilities (3, H, W) -> instance labels (H, W).

    Seeds are the connected regions of 'interior'. Each seed grows by watershed
    over the whole nucleus area (interior + boundary), so touching nuclei that
    the boundary class separated stay separate.
    """
    interior = prob[1] > thr
    nucleus = (prob[1] + prob[2]) > thr
    seeds = remove_small(ndi.label(interior)[0], min_size // 3 + 1)
    label = watershed(-prob[1], seeds, mask=nucleus)
    return remove_small(label, min_size)


def match(true, pred):
    """IoU between every true and every predicted nucleus, plus the overlap counts."""
    true = relabel_sequential(true)[0]
    pred = relabel_sequential(pred)[0]
    nt, npred = int(true.max()), int(pred.max())
    inter = np.bincount(true.ravel().astype(np.int64) * (npred + 1) + pred.ravel(),
                        minlength=(nt + 1) * (npred + 1)).reshape(nt + 1, npred + 1)
    area_t = inter.sum(1)[1:]
    area_p = inter.sum(0)[1:]
    inter = inter[1:, 1:]
    union = area_t[:, None] + area_p[None, :] - inter
    iou = inter / np.maximum(union, 1)
    return iou, inter, area_t, area_p


def dsb_scores(iou, thresholds=np.arange(0.5, 1.0, 0.05)):
    """Data Science Bowl 2018 score for one image and F1 at IoU 0.5.

    Above IoU 0.5 a nucleus can match at most one prediction, so counting
    rows and columns with a match is the same as a one-to-one matching.
    """
    nt, npred = iou.shape
    per_t = []
    for t in thresholds:
        hit = iou > t
        tp = int(hit.any(1).sum())
        fn = nt - tp
        fp = npred - int(hit.any(0).sum())
        per_t.append(tp / (tp + fp + fn) if tp + fp + fn else 1.0)
    hit = iou > 0.5
    tp = int(hit.any(1).sum())
    fn, fp = nt - tp, npred - int(hit.any(0).sum())
    f1 = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0
    return float(np.mean(per_t)), f1, tp, fp, fn


def merges_and_splits(inter, area_t, area_p, share=0.5):
    """Merge: one prediction covers more than half of two or more true nuclei.
    Split: one true nucleus covers more than half of two or more predictions."""
    if inter.size == 0:
        return 0, 0
    cover_t = inter / np.maximum(area_t[:, None], 1)
    cover_p = inter / np.maximum(area_p[None, :], 1)
    merges = int(((cover_t > share).sum(0) >= 2).sum())
    splits = int(((cover_p > share).sum(1) >= 2).sum())
    return merges, splits


def entropy_map(prob):
    p = np.clip(prob, 1e-8, 1.0)
    return -(p * np.log(p)).sum(0)


def warning_test(uncertainty, score, share=0.1):
    """Flag the most uncertain images and count how many of the worst images are caught."""
    from scipy.stats import hypergeom, spearmanr
    n = len(score)
    k = max(1, int(np.ceil(share * n)))
    worst = set(np.argsort(score)[:k])
    flagged = set(np.argsort(-np.asarray(uncertainty))[:k])
    caught = len(worst & flagged)
    expected = k * k / n
    p_value = float(hypergeom.sf(caught - 1, n, k, k))
    rho, rho_p = spearmanr(uncertainty, score)
    return dict(n_images=n, k=k, caught=caught, expected_by_chance=round(expected, 2),
                p_value=p_value, spearman_rho=float(rho), spearman_p=float(rho_p),
                missed=sorted(int(i) for i in worst - flagged))

## Run Cellpose-SAM on every validation image

One pretrained model, default settings, no tuning on our images. Tuning thresholds on the validation set would make the comparison unfair.

In [ ]:
model = models.CellposeModel(gpu=torch.cuda.is_available())

rows = []
for r in val.itertuples():
    img, true = load(paths[r.id])
    out = model.eval(img, batch_size=32, flow_threshold=0.4, cellprob_threshold=0.0)
    pred = remove_small(np.asarray(out[0]).astype(np.int32), 15)
    iou, inter, area_t, area_p = match(true, pred)
    ap, f1, tp, fp, fn = dsb_scores(iou)
    merges, splits = merges_and_splits(inter, area_t, area_p)
    rows.append(dict(id=r.id, type=r.type, n_true=int(true.max()), n_pred=int(pred.max()),
                     ap=ap, f1_50=f1, tp=tp, fp=fp, fn=fn, merges=merges, splits=splits))
cp = pd.DataFrame(rows)
cp.to_csv(OUT / 'per_image_cellpose.csv', index=False)

by_type = cp.groupby('type').agg(images=('id', 'size'), mean_ap=('ap', 'mean'), median_ap=('ap', 'median'),
                                 mean_f1_50=('f1_50', 'mean'), merges=('merges', 'sum'), splits=('splits', 'sum'),
                                 missed=('fn', 'sum'), extra=('fp', 'sum')).round(3)
by_type.loc['all'] = [len(cp), cp.ap.mean(), cp.ap.median(), cp.f1_50.mean(),
                      cp.merges.sum(), cp.splits.sum(), cp.fn.sum(), cp.fp.sum()]
by_type = by_type.round(3)
by_type.to_csv(OUT / 'by_type_cellpose.csv')
print(by_type.to_string())

## Compare with our U-Net (only if the output of notebook 03 was added)

Same images, so the comparison is paired: for every image we look at AP(U-Net) minus AP(Cellpose-SAM). A Wilcoxon signed-rank test per image type says whether the difference is larger than chance.

In [ ]:
from scipy.stats import wilcoxon

found = sorted(glob.glob('/kaggle/input/**/per_image.csv', recursive=True))
if not found:
    print('per_image.csv from notebook 03 not found - add it with + Add Input to compare.')
else:
    unet = pd.read_csv(found[0])[['id', 'ap', 'f1_50']].rename(columns={'ap': 'ap_unet', 'f1_50': 'f1_unet'})
    both = cp.merge(unet, on='id').rename(columns={'ap': 'ap_cellpose', 'f1_50': 'f1_cellpose'})
    both['diff'] = both.ap_unet - both.ap_cellpose
    rows = []
    for t, g in list(both.groupby('type')) + [('all', both)]:
        p = wilcoxon(g.ap_unet, g.ap_cellpose).pvalue if len(g) > 5 and (g['diff'] != 0).any() else np.nan
        rows.append(dict(type=t, images=len(g), ap_unet=g.ap_unet.mean(), ap_cellpose=g.ap_cellpose.mean(),
                         unet_better=int((g['diff'] > 0).sum()), cellpose_better=int((g['diff'] < 0).sum()), p_value=p))
    compare = pd.DataFrame(rows).round(4)
    compare.to_csv(OUT / 'compare.csv', index=False)
    print(compare.to_string(index=False))

    fig, ax = plt.subplots(figsize=(5, 5))
    for t in sorted(both.type.unique()):
        s = both[both.type == t]
        ax.scatter(s.ap_cellpose, s.ap_unet, s=14, label=t)
    ax.plot([0, 1], [0, 1], color='gray', lw=1)
    ax.set_xlabel('AP per image, Cellpose-SAM (no training)'); ax.set_ylabel('AP per image, our U-Net')
    ax.legend(); ax.set_title('Same images, two models')
    plt.tight_layout(); plt.savefig(OUT / 'unet_vs_cellpose.png', dpi=120); plt.show()

## What this answers

Write it in your own words after looking at the numbers:

- On which image types does the pretrained model without our data already match or beat the U-Net?
- Where does training on our own data still help, and how much?
- Which errors (merged or split nuclei) does each model make?

In [ ]:
shutil.make_archive(str(WORK / 'results_03b'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))